In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Sử dụng Qwen2.5-7B-Instruct - một trong những dòng 7B mạnh nhất hiện tại về tiếng Việt và JSON
model_id = "Qwen/Qwen2.5-7B-Instruct"

# Cấu hình nén 4-bit giúp mô hình chạy mượt mà, tốn ít VRAM trên T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print("--- Đang tải mô hình Qwen 7B (Hiểu tiếng Việt vượt trội) ---")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
print("=> Đã tải xong Qwen-7B thành công và sẵn sàng!")

In [ ]:
import pandas as pd

# File bạn đã tải lên Kaggle
file_path = '/kaggle/input/datasets/richarddoan/input-data/benchmark_descriptions_teacher.csv' 
df = pd.read_csv(file_path)

print(f"Đã nạp file thành công! Kích thước: {df.shape}")

In [ ]:
import json
import re
from tqdm import tqdm

tien_ich_cols = [
    'hem_xe_hoi', 'gan_cho_sieu_thi', 'gan_truong_hoc', 
    'gan_benh_vien', 'gan_cong_vien_ho_nuoc'
]

def extract_features_qwen(description):
    # Định nghĩa phân vai rõ ràng cho Qwen để ép đầu ra chuẩn JSON
    system_prompt = "Bạn là trợ lý AI chuyên trích xuất thông tin bất động sản chính xác và trả về định dạng JSON."
    
    user_prompt = f"""Đọc đoạn mô tả bất động sản sau và xác định xem các tiện ích xung quanh có được nhắc đến hay không.
Trả về giá trị 1 nếu CÓ nhắc đến (hoặc có từ viết tắt tương đương như: hxh, hẻm xe hơi, st, siêu thị, bv, bệnh viện, trg, trường học, cv, công viên), và 0 nếu KHÔNG nhắc đến.

BẮT BUỘC TRẢ VỀ ĐÚNG ĐỊNH DẠNG JSON VỚI CÁC KEY SAU (Tuyệt đối không giải thích, không viết thêm lời thoại nào ngoài JSON):
{{
    "hem_xe_hoi": 0,
    "gan_cho_sieu_thi": 0,
    "gan_truong_hoc": 0,
    "gan_benh_vien": 0,
    "gan_cong_vien_ho_nuoc": 0
}}

Mô tả: "{description}" """

    # Sử dụng apply_chat_template chuẩn chỉnh của Qwen
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs, 
            max_new_tokens=150, 
            do_sample=False # Chế độ logic cố định, tối ưu cho trích xuất JSON
        )
        
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    
    try:
        # Bóc tách chuỗi JSON bằng Regex đề phòng text rác thừa
        if "{" in response:
            json_clean = re.search(r'\{.*\}', response, re.DOTALL).group()
            return json.loads(json_clean)
        return None
    except Exception as e:
        # Nếu dòng nào lỗi, trả về mặc định toàn 0 để không dừng vòng lặp
        return {col: 0 for col in tien_ich_cols}

In [ ]:
# --- TIẾN HÀNH CHẠY VÒNG LẶP VỚI QWEN ---
print("Qwen đang tiến hành quét dữ liệu mô tả...")

for index, row in tqdm(df.iterrows(), total=len(df), desc="Xử lý"):
    description = row['Mô tả']
    
    if pd.isna(description):
        continue
        
    extracted_data = extract_features_qwen(description)
    
    if extracted_data:
        for col in tien_ich_cols:
            df.at[index, col] = extracted_data.get(col, 0)

# Lưu kết quả ra thư mục làm việc của Kaggle với tên mới
output_file = 'benchmark_ground_truth_qwen.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"\n=> Hoàn thành! File lưu tại: /kaggle/working/{output_file}")